# 추가 검증과 실행 목적의 확인

현재 습도 모형은 v10을 유지합니다. 추가36적합은 두 제외 검증의 모든 기준을 통과하지 못했습니다.
F1 우선 프로필은 이미 알려진 기온 기준선을 명시한 대안으로, 새 독립 모델 개선이 아닙니다.
GUI는 관측 정확도·결빙확률을 뜻하지 않는 구간 추정과 입력 품질을 검토합니다.

In [ ]:
from pathlib import Path
import json, hashlib, subprocess, sys, uuid
import pandas as pd
from IPython.display import display, Image
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'extended_training_plan.json').exists() or (p/'config/weather_catboost_20261101.json').exists())
PACKAGED=(ROOT/'extended_training_plan.json').exists()
P=ROOT/'research/latest' if PACKAGED else ROOT/'outputs/private'
read=lambda p:json.loads(p.read_text(encoding='utf8'))
digest=lambda p:hashlib.sha256(p.read_bytes()).hexdigest()
for name in ['weather_thermodynamics_20261101_v1','weather_catboost_20261101_v1','frost_icing_20261101_v1']:
    audit=read(P/name/'independent_verification.json')
    assert audit['status']=='passed' and audit['summary_sha256']==digest(P/name/'summary.json')
table=pd.read_csv(P/'followup_review_20261101_v1/weather_comparison_metrics.csv')
assert len(table)==80
display(table.loc[table.scope.eq('pooled')])
display(Image(filename=str(ROOT/'docs/images/weather_followup_20261101.png')))

In [ ]:
profile=read(P/'competition_profile_review_20261101_v1/summary.json')
assert profile['status']=='explicit_competition_profiles_verified' and profile['protected_missing_rows_preserved']==18
display(pd.read_csv(P/'followup_review_20261101_v1/icing_profile_metrics.csv'))
display(Image(filename=str(ROOT/'docs/images/competition_profiles_20261101.png')))
print('겨울별 반례·정밀도·재현율·사건과 함께 판단합니다. 공식 점수가 아닙니다.')

In [ ]:
viewer=P/'weather_explorer_20261101_v2'
assert viewer.joinpath('index.html').exists()
proof=read(P/'weather_explorer_validation_20261101_v4/summary.json')
assert proof['status']=='passed' and proof['html_sha256']==digest(viewer/'index.html')
assert not proof['network_requests'] and not proof['page_errors']
display(Image(filename=str(ROOT/'docs/images/weather_explorer_20261101.png')))
print('브라우저에서 열 파일:',viewer/'index.html')
print('구간 선택·두 겨울 비교·품질 필터·CSV 240행 다운로드를 검증했습니다.')

In [ ]:
bundle=ROOT if PACKAGED else ROOT/'outputs/private/submission_candidate_20261101_v10'
local=ROOT/'local_review' if PACKAGED else P/'followup_review_20261101_v1/local_review'
local.mkdir(parents=True,exist_ok=True)
output=local/('profile_'+uuid.uuid4().hex+'.csv')
command=[sys.executable,str(ROOT/'scripts/predict_competition_profile.py'),'--bundle',str(bundle),'--input',str(bundle/'data/weather_example.csv'),'--output',str(output),'--icing-profile','score_priority_asos_zero','--output-start','2024-12-15','--output-end','2024-12-16']
run=subprocess.run(command,cwd=ROOT,capture_output=True,text=True,encoding='utf8')
assert run.returncode==0,run.stdout+run.stderr
actual=pd.read_csv(output,dtype={'station':str,'predicted_ice':'Int64'},parse_dates=['time'],float_precision='round_trip')
expected=pd.read_csv(bundle/'data/example_predictions.csv',dtype={'station':str,'predicted_ice':'Int64'},parse_dates=['time'],float_precision='round_trip')
pd.testing.assert_series_equal(actual.estimated_road_temp_c,expected.estimated_road_temp_c,check_exact=True)
assert len(actual)==144 and actual.predicted_ice.isna().equals(expected.predicted_ice.isna())
assert actual.icing_policy.eq('score_priority_asos_zero_v1').all()
print('실제 F1 우선 CLI: 144행·온도·결측 보존 확인',output)